# WorkPulse — DistilBERT Training Notebook

Fine-tuning DistilBERT on 838K real Glassdoor reviews for 3-class company culture sentiment analysis.

**Dataset:** Gopinath-AI/glassdoor_reviews (HuggingFace)  
**Model:** distilbert-base-uncased → Madhuri1003/workpulse-distilbert  
**Final Accuracy:** 78.9% | F1 Macro: 0.789  

## Sections
1. Install dependencies
2. Dataset discovery and selection
3. Data loading and exploration
4. Label-text mismatch analysis (key insight)
5. Data validation and class quality check
6. Preprocessing and balanced sampling
7. Tokenization
8. Model setup and training
9. Evaluation and inference test
10. Aspect rating exploration (future work)
11. Push to HuggingFace Hub

## 1. Install Dependencies

In [1]:
!pip install transformers datasets accelerate scikit-learn -q

## 2. Dataset Discovery
Searching HuggingFace for real Glassdoor review datasets.

In [2]:
from huggingface_hub import list_datasets

results = list(list_datasets(search="glassdoor", limit=20))
for r in results:
    print(r.id)

dariadaria/glassdoor_reviews_gpt4_0
lallantop/glassdoor
jedha0padavan/glassdoor-reviews-tokenized
jedha0padavan/glassdoor_reviews_processed
Abdullah4747/Glassdoor_dataset_2017
Gopinath-AI/glassdoor_reviews


## 3. Dataset Comparison
Loading and comparing the top 3 candidate datasets to find the best one for our use case.

In [3]:
from datasets import load_dataset

# Check the most promising ones
candidates = [
    "dariadaria/glassdoor_reviews_gpt4_0",
    "Gopinath-AI/glassdoor_reviews",
    "Abdullah4747/Glassdoor_dataset_2017"
]

for name in candidates:
    try:
        ds = load_dataset(name, split="train[:5]")
        print(f"\n{'='*50}")
        print(f"Dataset: {name}")
        print(f"Columns: {ds.column_names}")
        print(f"Sample row: {ds[0]}")
    except Exception as e:
        print(f"\n{name} -> ERROR: {e}")

README.md:   0%|          | 0.00/1.50k [00:00<?, ?B/s]

data/train-00000-of-00001-16c43bc5650491(…):   0%|          | 0.00/147k [00:00<?, ?B/s]

data/test-00000-of-00001-cf848eff17e3e8a(…):   0%|          | 0.00/64.1k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/385 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/129 [00:00<?, ? examples/s]

...

Generating train split:   0%|          | 0/838566 [00:00<?, ? examples/s]


Dataset: Gopinath-AI/glassdoor_reviews
Columns: ['firm', 'date_review', 'job_title', 'current', 'location', 'overall_rating', 'work_life_balance', 'culture_values', 'diversity_inclusion', 'career_opp', 'comp_benefits', 'senior_mgmt', 'recommend', 'ceo_approv', 'outlook', 'headline', 'pros', 'cons']
Sample row: {'firm': 'AFH-Wealth-Management', 'date_review': '2015-04-05', 'job_title': ' ', 'current': 'Current Employee', 'location': None, 'overall_rating': 2, 'work_life_balance': 4.0, 'culture_values': 3.0, 'diversity_inclusion': None, 'career_opp': 2.0, 'comp_benefits': 3.0, 'senior_mgmt': 3.0, 'recommend': 'x', 'ceo_approv': 'o', 'outlook': 'r', 'headline': 'Young colleagues, poor micro management', 'pros': 'Very friendly and welcoming to new staff. Easy going ethic.', 'cons': 'Poor salaries, poor training and communication.'}


glassdoor_jobs.csv:   0%|          | 0.00/3.84M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/956 [00:00<?, ? examples/s]


Dataset: Abdullah4747/Glassdoor_dataset_2017
Columns: ['Unnamed: 0', 'Job Title', 'Salary Estimate', 'Job Description', 'Rating', 'Company Name', 'Location', 'Headquarters', 'Size', 'Founded', 'Type of ownership', 'Industry', 'Sector', 'Revenue', 'Competitors']
Sample row: {'Unnamed: 0': 0, 'Job Title': 'Data Scientist', 'Salary Estimate': '$53K-$91K (Glassdoor est.)', 'Job Description': 'Data Scientist\nLocation: Albuquerque, NM\nEducation Required: Bachelor’s degree required, preferably in math, engineering, business, or the sciences.\nSkills Required:\nBachelor’s Degree in relevant field, e.g., math, data analysis, database, computer science, Artificial Intelligence (AI); three years’ experience credit for Master’s degree; five years’ experience credit for a Ph.D\nApplicant should be proficient in the use of Power BI, Tableau, Python, MATLAB, Microsoft Word, PowerPoint, Excel, and working knowledge of MS Access, LMS, SAS, data visualization tools, and have a strong algorithmic apti

## 4. Data Loading and Exploration
We selected `Gopinath-AI/glassdoor_reviews` — 838K real Glassdoor reviews with pros, cons, and star ratings.

In [4]:
from datasets import load_dataset
import pandas as pd

# Load full dataset
dataset = load_dataset("Gopinath-AI/glassdoor_reviews")
df = dataset["train"].to_pandas()

print("Shape:", df.shape)
print("\nColumn types:\n", df.dtypes)
print("\nMissing values:\n", df.isnull().sum())
print("\nRating distribution:\n", df["overall_rating"].value_counts().sort_index())
print("\nSample pros:", df["pros"].iloc[0])
print("Sample cons:", df["cons"].iloc[0])
print("Sample rating:", df["overall_rating"].iloc[0])

Shape: (838566, 18)

Column types:
 firm                    object
date_review             object
job_title               object
current                 object
location                object
overall_rating           int64
work_life_balance      float64
culture_values         float64
diversity_inclusion    float64
career_opp             float64
comp_benefits          float64
senior_mgmt            float64
recommend               object
ceo_approv              object
outlook                 object
headline                object
pros                    object
cons                    object
dtype: object

Missing values:
 firm                        0
date_review                 0
job_title                   0
current                     0
location               297343
overall_rating              0
work_life_balance      149894
culture_values         191373
diversity_inclusion    702500
career_opp             147501
comp_benefits          150082
senior_mgmt            155876
recommend     

## 5. Text Length and Class Distribution Analysis
Checking text lengths against DistilBERT's 512 token limit, and mapping 5-star ratings to 3 classes.

In [5]:
import pandas as pd
from datasets import load_dataset

dataset = load_dataset("Gopinath-AI/glassdoor_reviews")
df = dataset["train"].to_pandas()

# Step 1: Check actual text lengths — DistilBERT has 512 token limit
df["combined_text"] = df["pros"].fillna("") + " [SEP] " + df["cons"].fillna("")
df["text_length"] = df["combined_text"].apply(lambda x: len(x.split()))

print("=== TEXT LENGTH STATS ===")
print(df["text_length"].describe())
print(f"\nRows exceeding 400 words (will be truncated): {(df['text_length'] > 400).sum()}")
print(f"That's {(df['text_length'] > 400).mean()*100:.1f}% of data")

# Step 2: Map ratings to 3 classes
def map_label(r):
    if r <= 2: return 0   # Negative
    elif r == 3: return 1  # Neutral
    else: return 2         # Positive

df["label"] = df["overall_rating"].apply(map_label)

print("\n=== CLASS DISTRIBUTION AFTER MAPPING ===")
counts = df["label"].value_counts().sort_index()
labels = {0: "Negative", 1: "Neutral", 2: "Positive"}
for k, v in counts.items():
    print(f"{labels[k]}: {v} ({v/len(df)*100:.1f}%)")

# Step 3: Check if text is actually meaningful (not just 1-2 words)
print("\n=== SHORT TEXT SAMPLES (under 5 words) ===")
short = df[df["text_length"] < 5][["pros", "cons", "overall_rating"]].head(10)
print(short.to_string())

# Step 4: Sample across each class so you can eyeball quality
print("\n=== RANDOM SAMPLE PER CLASS ===")
for label in [0, 1, 2]:
    sample = df[df["label"] == label].sample(2, random_state=42)[["combined_text", "overall_rating"]]
    print(f"\n-- {labels[label]} --")
    for _, row in sample.iterrows():
        print(f"  Rating {row['overall_rating']}: {row['combined_text'][:200]}")

=== TEXT LENGTH STATS ===
count    838566.000000
mean         36.903743
std          50.745779
min           4.000000
25%          14.000000
50%          20.000000
75%          41.000000
max        3208.000000
Name: text_length, dtype: float64

Rows exceeding 400 words (will be truncated): 2427
That's 0.3% of data

=== CLASS DISTRIBUTION AFTER MAPPING ===
Negative: 133767 (16.0%)
Neutral: 194267 (23.2%)
Positive: 510532 (60.9%)

=== SHORT TEXT SAMPLES (under 5 words) ===
                         pros          cons  overall_rating
177173  diverse opportunities  compensation               5

=== RANDOM SAMPLE PER CLASS ===

-- Negative --
  Rating 1: Pay is good and people are generally nice. [SEP] Beware of Leadership! Culture is terrible.
  Rating 1: Having the largest Big 4 brand on your resume. [SEP] I've worked for 2 of the Big 4 Firms and this was a terrible experience.  They will misled you with false expectations, promotions, salary and hour

-- Neutral --
  Rating 3: Interesting

## 6. Label-Text Mismatch Discovery ⚠️
**Key insight:** 46.2% of negative reviews (rating 1-2) contain positive language in their `pros` field.  
Combining pros+cons as input would teach the model that positive language = negative sentiment.  
**Solution:** Use `cons` only for Negative/Neutral labels, `pros` only for Positive labels.

In [6]:
# Verify the label-text mismatch theory with real numbers
print("=== FOR RATING 1-2 (Negative): How positive does pros text look? ===")
neg_df = df[df["overall_rating"] <= 2].copy()

# Simple keyword check — not ML, just sanity check
positive_words = ["great", "good", "excellent", "amazing", "love", "best", "wonderful", "fantastic"]
negative_words = ["bad", "poor", "terrible", "awful", "worst", "toxic", "horrible", "avoid"]

def count_keywords(text, words):
    text = str(text).lower()
    return sum(1 for w in words if w in text)

neg_df["pos_words_in_pros"] = neg_df["pros"].apply(lambda x: count_keywords(x, positive_words))
neg_df["neg_words_in_pros"] = neg_df["pros"].apply(lambda x: count_keywords(x, negative_words))

print(f"Rating 1-2 reviews where PROS contains positive words: {(neg_df['pos_words_in_pros'] > 0).sum()} / {len(neg_df)}")
print(f"That's {(neg_df['pos_words_in_pros'] > 0).mean()*100:.1f}% of negative-rated reviews")

print("\n=== FOR RATING 4-5 (Positive): How negative does cons text look? ===")
pos_df = df[df["overall_rating"] >= 4].copy()
pos_df["neg_words_in_cons"] = pos_df["cons"].apply(lambda x: count_keywords(x, negative_words))
print(f"Rating 4-5 reviews where CONS contains negative words: {(pos_df['neg_words_in_cons'] > 0).sum()} / {len(pos_df)}")
print(f"That's {(pos_df['neg_words_in_cons'] > 0).mean()*100:.1f}% of positive-rated reviews")

print("\n=== DECISION POINT: Average pos_words in pros by rating ===")
df["pos_in_pros"] = df["pros"].apply(lambda x: count_keywords(x, positive_words))
df["neg_in_cons"] = df["cons"].apply(lambda x: count_keywords(x, negative_words))
print(df.groupby("overall_rating")[["pos_in_pros", "neg_in_cons"]].mean().round(3))

=== FOR RATING 1-2 (Negative): How positive does pros text look? ===
Rating 1-2 reviews where PROS contains positive words: 61787 / 133767
That's 46.2% of negative-rated reviews

=== FOR RATING 4-5 (Positive): How negative does cons text look? ===
Rating 4-5 reviews where CONS contains negative words: 27442 / 510532
That's 5.4% of positive-rated reviews

=== DECISION POINT: Average pos_words in pros by rating ===
                pos_in_pros  neg_in_cons
overall_rating                          
1                     0.445        0.422
2                     0.600        0.284
3                     0.664        0.142
4                     0.757        0.068
5                     0.814        0.041


## 7. Validating the Label-Text Strategy
Verifying that cons text for negative reviews and pros text for positive reviews is semantically consistent.

In [7]:
# Verify cons quality for negative reviews
print("=== CONS TEXT FOR RATING 1-2 (what we'll actually train on) ===")
samples = df[df["overall_rating"] <= 2]["cons"].dropna().sample(8, random_state=42)
for i, text in enumerate(samples):
    print(f"\n{i+1}. {text[:200]}")

print("\n\n=== PROS TEXT FOR RATING 4-5 (what we'll actually train on) ===")
samples = df[df["overall_rating"] >= 4]["pros"].dropna().sample(8, random_state=42)
for i, text in enumerate(samples):
    print(f"\n{i+1}. {text[:200]}")

print("\n\n=== CONS TEXT FOR RATING 3 (neutral) ===")
samples = df[df["overall_rating"] == 3]["cons"].dropna().sample(5, random_state=42)
for i, text in enumerate(samples):
    print(f"\n{i+1}. {text[:200]}")

# Also check: after this strategy, what are our class sizes?
print("\n\n=== FINAL CLASS SIZES WITH NEW STRATEGY ===")
neg = df[df["overall_rating"] <= 2]["cons"].dropna()
neu = df[df["overall_rating"] == 3]["cons"].dropna()
pos = df[df["overall_rating"] >= 4]["pros"].dropna()

print(f"Negative samples (cons, rating 1-2): {len(neg):,}")
print(f"Neutral  samples (cons, rating 3):   {len(neu):,}")
print(f"Positive samples (pros, rating 4-5): {len(pos):,}")
print(f"Total: {len(neg)+len(neu)+len(pos):,}")
print(f"\nImbalance ratio (pos/neg): {len(pos)/len(neg):.1f}x")

=== CONS TEXT FOR RATING 1-2 (what we'll actually train on) ===

1. Beware of Leadership! Culture is terrible.

2. I've worked for 2 of the Big 4 Firms and this was a terrible experience.  They will misled you with false expectations, promotions, salary and hours.  The Partners will never put anything in writing s

3. Selling aspect Of role with no incentives

4. Let me just state that the DPA of 2012 wasn't surprising given the far flung, year after year shenanigans at the bank; the feds velvet hammer triggered an internal shotgun response where a scalpel was

5. Bully culture created by senior management

6. many procedure have to follow

7. You're in the mortgage business, which isn't exactly loved by most Americans these days. Mortgage industry is in a lull and the company reflects the reality of the market. Job is on the line from a qu

8. low salaries, cannot trust coworkers


=== PROS TEXT FOR RATING 4-5 (what we'll actually train on) ===

1. Great employee benefit, paid day-off

## 8. Preprocessing and Balanced Sampling
Building the final training dataset:
- Negative (rating 1-2): cons text only
- Neutral (rating 3): cons text only  
- Positive (rating 4-5): pros text only
- Undersample to 16,666 per class → 50K total balanced dataset

In [8]:
import pandas as pd
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split

# Load
dataset = load_dataset("Gopinath-AI/glassdoor_reviews")
df = dataset["train"].to_pandas()

# Build clean, label-matched samples
neg = df[df["overall_rating"] <= 2][["cons"]].dropna().copy()
neg["text"] = neg["cons"]
neg["label"] = 0

neu = df[df["overall_rating"] == 3][["cons"]].dropna().copy()
neu["text"] = neu["cons"]
neu["label"] = 1

pos = df[df["overall_rating"] >= 4][["pros"]].dropna().copy()
pos["text"] = pos["pros"]
pos["label"] = 2

# Undersample to 16,666 each → 50K total balanced
SAMPLE = 16_666
neg_s = neg.sample(SAMPLE, random_state=42)
neu_s = neu.sample(SAMPLE, random_state=42)
pos_s = pos.sample(SAMPLE, random_state=42)

final_df = pd.concat([neg_s, neu_s, pos_s])[["text", "label"]].reset_index(drop=True)
final_df = final_df.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle

# Verify
print("=== FINAL DATASET ===")
print(f"Total rows: {len(final_df)}")
print(f"Class distribution:\n{final_df['label'].value_counts().sort_index()}")
print(f"\nLabel map: 0=Negative, 1=Neutral, 2=Positive")
print(f"\nSample rows:")
print(final_df.sample(6, random_state=42)[["text", "label"]].to_string())

# Check for empty/whitespace text
final_df["text"] = final_df["text"].str.strip()
empty = (final_df["text"] == "").sum()
print(f"\nEmpty text rows: {empty}")

# Train/val split
train_df, val_df = train_test_split(final_df, test_size=0.2,
                                     random_state=42, stratify=final_df["label"])
print(f"\nTrain size: {len(train_df)}")
print(f"Val size:   {len(val_df)}")
print(f"\nTrain class distribution:\n{train_df['label'].value_counts().sort_index()}")
print(f"Val class distribution:\n{val_df['label'].value_counts().sort_index()}")

=== FINAL DATASET ===
Total rows: 49998
Class distribution:
label
0    16666
1    16666
2    16666
Name: count, dtype: int64

Label map: 0=Negative, 1=Neutral, 2=Positive

Sample rows:
                                                                                                                                                                                                                                                                                                                                                                                                                  text  label
33552  Killer benefits package. Reasonably priced, on-site food (Cafe Mac). Employee discount. Awesome corporate culture (quarterly beer bashes, no issues with visible tattoos/piercings, non-existent dress code, okay to be an individual and not a business-casual drone. Occasional opportunity to do something incredible that can change the world. GLBTQ-friendly. Many options to work-from-home/telecom

## 9. Tokenization
Using DistilBERT tokenizer with max_length=128 (sufficient for median text length of 20 words).

In [9]:
# Cell: Tokenization
from transformers import AutoTokenizer
import torch
from torch.utils.data import Dataset as TorchDataset

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Test tokenizer on one sample first
sample = "Bully culture, no growth, terrible management"
tokens = tokenizer(sample, truncation=True, max_length=128, padding="max_length")
print("Input IDs length:", len(tokens["input_ids"]))
print("Tokens:", tokenizer.convert_ids_to_tokens(tokens["input_ids"])[:15])
print("\nTokenizer working correctly.")

# Build PyTorch Dataset
class ReviewDataset(TorchDataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.texts = df["text"].tolist()
        self.labels = df["label"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = ReviewDataset(train_df, tokenizer)
val_dataset   = ReviewDataset(val_df, tokenizer)

print(f"\nTrain dataset size: {len(train_dataset)}")
print(f"Val dataset size:   {len(val_dataset)}")

# Verify one batch shape
sample_item = train_dataset[0]
print(f"\nSample input_ids shape:      {sample_item['input_ids'].shape}")
print(f"Sample attention_mask shape: {sample_item['attention_mask'].shape}")
print(f"Sample label:                {sample_item['labels']}")
print("\nDataset ready.")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Input IDs length: 128
Tokens: ['[CLS]', 'bully', 'culture', ',', 'no', 'growth', ',', 'terrible', 'management', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']

Tokenizer working correctly.

Train dataset size: 39998
Val dataset size:   10000

Sample input_ids shape:      torch.Size([128])
Sample attention_mask shape: torch.Size([128])
Sample label:                1

Dataset ready.


## 10. Model Setup
Loading distilbert-base-uncased with a new 3-class classification head.  
UNEXPECTED keys = MLM head from pre-training (discarded).  
MISSING keys = new classification head (randomly initialized, will be trained).

In [10]:
# Cell: Model + Training
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

# Load model — 3 labels
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label={0: "Negative", 1: "Neutral", 2: "Positive"},
    label2id={"Negative": 0, "Neutral": 1, "Positive": 2}
)

# Metrics function — accuracy + per-class F1
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average=None)  # per class
    f1_macro = f1_score(labels, predictions, average="macro")
    return {
        "accuracy": round(acc, 4),
        "f1_negative": round(f1[0], 4),
        "f1_neutral":  round(f1[1], 4),
        "f1_positive": round(f1[2], 4),
        "f1_macro":    round(f1_macro, 4),
    }

# Training arguments — tuned for Colab free GPU
training_args = TrainingArguments(
    output_dir="./workpulse-distilbert",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_steps=200,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    fp16=True,                          # faster on Colab GPU
    report_to="none"                    # no wandb
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("Model loaded. Starting training...")
print(f"Total training steps: {len(train_dataset) // 32 * 3}")
print(f"Device: {'GPU ✅' if torch.cuda.is_available() else 'CPU ⚠️ — go to Runtime > Change runtime type > T4 GPU'}")



model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Model loaded. Starting training...
Total training steps: 3747
Device: GPU ✅


## 11. Training
Training for 3 epochs on T4 GPU (~8 minutes).  
Best model selected by F1 Macro on validation set.

In [11]:
# Cell: Train
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1 Negative,F1 Neutral,F1 Positive,F1 Macro
1,0.460095,0.453015,0.785900,0.696700,0.695300,0.963000,0.785000
2,0.386012,0.451975,0.791000,0.684200,0.719500,0.965700,0.789800
3,0.263977,0.525799,0.781300,0.679300,0.698800,0.966100,0.781400


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3750, training_loss=0.39312871119181314, metrics={'train_runtime': 459.4248, 'train_samples_per_second': 261.183, 'train_steps_per_second': 8.162, 'total_flos': 3973894126078464.0, 'train_loss': 0.39312871119181314, 'epoch': 3.0})

## 12. Inference Test
Testing the trained model on 5 representative reviews before pushing to HuggingFace Hub.

In [12]:
from huggingface_hub import login
import os

# Login with your token
login(token=os.environ.get("HF_TOKEN"))  # reads from Colab secrets

# Push model and tokenizer
# Replace YOUR_HF_USERNAME with your actual HuggingFace username
HF_USERNAME = "Madhuri1003"
REPO_NAME = "workpulse-distilbert"

print(f"Pushing to: {HF_USERNAME}/{REPO_NAME}")

model.push_to_hub(f"{HF_USERNAME}/{REPO_NAME}")
tokenizer.push_to_hub(f"{HF_USERNAME}/{REPO_NAME}")

print(f"\nModel live at: https://huggingface.co/{HF_USERNAME}/{REPO_NAME}")

Pushing to: Madhuri1003/workpulse-distilbert


README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...dg65ldm/model.safetensors:   2%|2         | 6.45MB /  268MB            


Model live at: https://huggingface.co/Madhuri1003/workpulse-distilbert


## 13. Push to HuggingFace Hub

In [13]:
from transformers import pipeline

# Load directly from hub after push completes
# For now test locally from saved model
classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0  # GPU
)

test_reviews = [
    "Toxic management, no work life balance, constant pressure and burnout",
    "Decent salary but no growth opportunities, management could be better",
    "Amazing culture, great benefits, supportive team and excellent work life balance",
    "They fire people without reason, politics everywhere, avoid this company",
    "Average company, nothing special but nothing terrible either",
]

print("=== WORKPULSE INFERENCE TEST ===\n")
for review in test_reviews:
    result = classifier(review)[0]
    label = result["label"]
    score = result["score"]
    bar = "█" * int(score * 20)
    print(f"Review: {review[:60]}...")
    print(f"Result: {label} ({score:.1%}) {bar}\n")

=== WORKPULSE INFERENCE TEST ===

Review: Toxic management, no work life balance, constant pressure an...
Result: Negative (93.1%) ██████████████████

Review: Decent salary but no growth opportunities, management could ...
Result: Positive (86.4%) █████████████████

Review: Amazing culture, great benefits, supportive team and excelle...
Result: Positive (99.9%) ███████████████████

Review: They fire people without reason, politics everywhere, avoid ...
Result: Negative (98.0%) ███████████████████

Review: Average company, nothing special but nothing terrible either...
Result: Neutral (66.2%) █████████████



## 14. Aspect Rating Exploration (Future Work)
The dataset contains per-aspect ratings (work_life_balance, culture_values, career_opp, comp_benefits, senior_mgmt).  
76% of reviews have all 5 aspects filled — sufficient for future aspect-based sentiment analysis.

In [14]:
import pandas as pd
from datasets import load_dataset

dataset = load_dataset("Gopinath-AI/glassdoor_reviews")
df = dataset["train"].to_pandas()

# Check aspect column quality
aspects = ["work_life_balance", "culture_values", "career_opp",
           "comp_benefits", "senior_mgmt"]

print("=== NULL COUNTS ===")
print(df[aspects].isnull().sum())

print("\n=== VALUE DISTRIBUTIONS ===")
for col in aspects:
    print(f"\n{col}:")
    print(df[col].value_counts().sort_index())

print("\n=== SAMPLE WITH ALL ASPECTS FILLED ===")
complete = df.dropna(subset=aspects)
print(f"Rows with all 5 aspects filled: {len(complete)} / {len(df)}")
print(f"That's {len(complete)/len(df)*100:.1f}% of data")

print("\n=== SAMPLE ROW ===")
sample = complete.sample(3, random_state=42)[["pros", "cons"] + aspects]
print(sample.to_string())

=== NULL COUNTS ===
work_life_balance    149894
culture_values       191373
career_opp           147501
comp_benefits        150082
senior_mgmt          155876
dtype: int64

=== VALUE DISTRIBUTIONS ===

work_life_balance:
work_life_balance
1.0     81481
2.0     92801
3.0    169056
4.0    176147
5.0    169187
Name: count, dtype: int64

culture_values:
culture_values
1.0     70813
2.0     65904
3.0    131024
4.0    169424
5.0    210028
Name: count, dtype: int64

career_opp:
career_opp
1.0     69977
2.0     85349
3.0    168571
4.0    189882
5.0    177286
Name: count, dtype: int64

comp_benefits:
comp_benefits
1.0     62336
2.0     93101
3.0    189896
4.0    194437
5.0    148714
Name: count, dtype: int64

senior_mgmt:
senior_mgmt
1.0    108354
2.0     98302
3.0    171941
4.0    172629
5.0    131464
Name: count, dtype: int64

=== SAMPLE WITH ALL ASPECTS FILLED ===
Rows with all 5 aspects filled: 637409 / 838566
That's 76.0% of data

=== SAMPLE ROW ===
                                       

## Summary

| Metric | Value |
|---|---|
| Dataset | 838,566 Glassdoor reviews |
| Training samples | 50,000 (balanced) |
| Model | distilbert-base-uncased |
| Overall Accuracy | 78.9% |
| F1 — Negative | 0.689 |
| F1 — Neutral | 0.712 |
| F1 — Positive | 0.966 |
| F1 — Macro | 0.789 |

**Model live at:** https://huggingface.co/Madhuri1003/workpulse-distilbert  
**Demo:** https://workpulse-frontend-pied.vercel.app